# MedCLIP-SAMv2 — Sheffield Evaluation — GPU Lambda

GPU-accelerated evaluation of MedCLIP-SAMv2 segmentations on the Sheffield dataset.

- **Predictions**: `~/medclipsamv2_sheffield_segs/Aug_N_medclipsamv2.npz` (L_/R_ prefix keys)
- **GT**: `~/sheffeld/20440203/Aug_N_segmentations.dcm` (labels 1–37, bilateral)

## Upload to Lambda
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/sheffeld/20440203/ \
  ubuntu@<IP>:~/sheffeld/20440203/

rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medclipsamv2/sheffield_segs/ \
  ubuntu@<IP>:~/medclipsamv2_sheffield_segs/
```

## Download results
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<IP>:~/medclipsamv2_sheffield_results/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medclipsamv2/codes/results_sheffield/
```

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'pydicom', 'SimpleITK', 'pandas', 'numpy<2'])
print('Dependencies ready.')

In [ ]:
import glob, os, re
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn.functional as F

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  {torch.cuda.get_device_name(0)}')
    free, total = torch.cuda.mem_get_info(0)
    print(f'  VRAM: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')

In [ ]:
def _to_bool(arr, device):
    return torch.from_numpy(arr.astype(np.uint8)).bool().to(device)

def _erode3d(mask):
    inv = (~mask).float().view(1, 1, *mask.shape)
    inv_pad = F.pad(inv, [1, 1, 1, 1, 1, 1], value=0)
    return ~(F.max_pool3d(inv_pad, kernel_size=3, stride=1, padding=0).squeeze(0).squeeze(0) > 0)

def _dilate3d(mask, dist=1):
    m = mask.float().view(1, 1, *mask.shape)
    for _ in range(dist):
        m = F.max_pool3d(F.pad(m, [1,1,1,1,1,1], value=0), kernel_size=3, stride=1, padding=0)
    return m.squeeze(0).squeeze(0) > 0

def _surface3d(mask):
    return mask & ~_erode3d(mask)

def _surface_pts(mask, spacing):
    surf = _surface3d(mask)
    if surf.sum() == 0: return None
    idx = surf.nonzero(as_tuple=False).float()
    scale = torch.tensor([spacing[2], spacing[1], spacing[0]], dtype=torch.float32, device=mask.device)
    return idx * scale

def hausdorff_gpu(pred, gt, spacing, chunk_a=512, chunk_b=512):
    # Both sides chunked — a single un-chunked side blows up memory to
    # O(chunk_a * len(b)) when a mask is noisy/degenerate (e.g. near-volume-
    # wide false positives turn most voxels into "surface" voxels).
    if pred.sum() == 0 or gt.sum() == 0: return float('nan')
    pp, gp = _surface_pts(pred, spacing), _surface_pts(gt, spacing)
    if pp is None or gp is None: return float('nan')
    def directed_max(a, b):
        max_min = torch.tensor(0.0, device=a.device)
        for i in range(0, len(a), chunk_a):
            a_chunk   = a[i:i + chunk_a]
            min_dists = torch.full((len(a_chunk),), float('inf'), device=a.device)
            for j in range(0, len(b), chunk_b):
                d         = torch.cdist(a_chunk, b[j:j + chunk_b])
                min_dists = torch.minimum(min_dists, d.min(dim=1).values)
            max_min = torch.maximum(max_min, min_dists.max())
        return max_min.item()
    return max(directed_max(pp, gp), directed_max(gp, pp))

def dice_gpu(pred, gt):
    return (2*(pred&gt).float().sum() / (pred.float().sum()+gt.float().sum()+1e-8)).item()

def jaccard_gpu(pred, gt):
    return ((pred&gt).float().sum() / ((pred|gt).float().sum()+1e-8)).item()

def volume_similarity_gpu(pred, gt):
    ps, gs = pred.float().sum(), gt.float().sum()
    return (1-(ps-gs).abs()/(ps+gs+1e-8)).item()

def false_negative_gpu(pred, gt):
    return ((~pred&gt).float().sum()/(gt.float().sum()+1e-8)).item()

def false_positive_gpu(pred, gt):
    return ((pred&~gt).float().sum()/(pred.float().sum()+1e-8)).item()

def bce_gpu(pred, gt, eps=1e-7):
    pf = pred.float().clamp(eps,1-eps); gf = gt.float()
    return (-gf*pf.log()-(1-gf)*(1-pf).log()).mean().item()

def boundary_iou_3d_gpu(pred, gt, dist=1):
    bp = _dilate3d(_surface3d(pred), dist); bg = _dilate3d(_surface3d(gt), dist)
    return ((bp&bg).float().sum()/((bp|bg).float().sum()+1e-8)).item()

def inter_slice_dice_gpu(pred):
    if pred.shape[0] < 2: return float('nan')
    a, b = pred[:-1].float(), pred[1:].float()
    inter = (a*b).sum(dim=(1,2)); denom = a.sum(dim=(1,2))+b.sum(dim=(1,2))
    valid = denom > 0
    return (2*inter[valid]/denom[valid]).mean().item() if valid.any() else 0.0

print('GPU helpers ready.')

In [ ]:
def read_gt(idx):
    ds  = pydicom.dcmread(os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm'))
    raw = ds.pixel_array.astype(np.float32)
    if raw.ndim == 2: raw = raw[np.newaxis]
    labeled = np.round(raw * 37.0 / 255.0).astype(np.int32)
    labeled[raw == 0] = 0
    return np.clip(labeled, 0, 37)

def get_spacing(idx):
    ds = pydicom.dcmread(os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm'))
    ps = getattr(ds, 'PixelSpacing', [1.0, 1.0])
    st = float(getattr(ds, 'SliceThickness', 1.0))
    return (float(ps[1]), float(ps[0]), st)

print('DICOM helpers ready.')

In [ ]:
BOUNDARY_DISTANCE = 1
GT_DIR     = os.path.expanduser('~/sheffeld/20440203')
SEG_DIR    = os.path.expanduser('~/medclipsamv2_sheffield_segs')
RESULT_DIR = os.path.expanduser('~/medclipsamv2_sheffield_results')
ALGO_TAG   = 'medclipsamv2'
os.makedirs(RESULT_DIR, exist_ok=True)

# (muscle_name, sheffield_gt_label, npz_keys)
# biceps_femoris maps to GT label 5 (biceps_femoris_long) — model outputs combined.
MUSCLES = [
    ('adductor_magnus',    3,  ['L_adductor_magnus',    'R_adductor_magnus'   ]),
    ('biceps_femoris',     5,  ['L_biceps_femoris',     'R_biceps_femoris'    ]),
    ('gracilis',           16, ['L_gracilis',           'R_gracilis'          ]),
    ('rectus_femoris',     27, ['L_rectus_femoris',     'R_rectus_femoris'    ]),
    ('sartorius',          28, ['L_sartorius',          'R_sartorius'         ]),
    ('semimembranosus',    29, ['L_semimembranosus',    'R_semimembranosus'   ]),
    ('semitendinosus',     30, ['L_semitendinosus',     'R_semitendinosus'    ]),
    ('vastus_intermedius', 35, ['L_vastus_intermedius', 'R_vastus_intermedius']),
    ('vastus_lateralis',   36, ['L_vastus_lateralis',   'R_vastus_lateralis'  ]),
    ('vastus_medialis',    37, ['L_vastus_medialis',    'R_vastus_medialis'   ]),
]

seg_files = sorted(glob.glob(os.path.join(SEG_DIR, 'Aug_*_medclipsamv2.npz')),
                   key=lambda p: int(re.search(r'Aug_(\d+)', p).group(1)))
print(f'GT dir : {GT_DIR}')
print(f'Seg dir: {SEG_DIR}')
print(f'Found  : {len(seg_files)} NPZ files')
if seg_files:
    s = np.load(seg_files[0])
    print(f'Sample keys: {sorted(s.files)}')

In [ ]:
def load_pred(seg_path, npz_keys, gt_shape):
    npz_data = np.load(seg_path)
    pred_np  = np.zeros(gt_shape, dtype=np.uint8)
    for key in npz_keys:
        if key in npz_data.files:
            pred_np |= npz_data[key].astype(np.uint8)
        else:
            print(f'    missing key "{key}"')
    return pred_np


def evaluate_muscle(muscle_name, sheffield_label, npz_keys):
    results = []
    for seg_path in seg_files:
        m = re.search(r'Aug_(\d+)', seg_path.replace('\\', '/'))
        if not m: continue
        idx     = m.group(1)
        gt_path = os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm')
        if not os.path.exists(gt_path):
            print(f'  [skip] GT missing: Aug_{idx}'); continue
        gt_arr  = read_gt(idx)
        spacing = get_spacing(idx)
        gt_bin  = (gt_arr == sheffield_label).astype(np.uint8)
        pred_np = load_pred(seg_path, npz_keys, gt_arr.shape)
        pred_t  = _to_bool(pred_np, DEVICE)
        gt_t    = _to_bool(gt_bin,  DEVICE)
        with torch.no_grad():
            row = {
                'sample':                               f'Aug_{idx}',
                f'{muscle_name}_dice':                  dice_gpu(pred_t, gt_t),
                f'{muscle_name}_hausdorff':             hausdorff_gpu(pred_t, gt_t, spacing),
                f'{muscle_name}_jaccard':               jaccard_gpu(pred_t, gt_t),
                f'{muscle_name}_volume_similarity':     volume_similarity_gpu(pred_t, gt_t),
                f'{muscle_name}_false_negative':        false_negative_gpu(pred_t, gt_t),
                f'{muscle_name}_false_positive':        false_positive_gpu(pred_t, gt_t),
                f'{muscle_name}_bce':                   bce_gpu(pred_t, gt_t),
                f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d_gpu(pred_t, gt_t, BOUNDARY_DISTANCE),
                f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice_gpu(pred_t),
                f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice_gpu(gt_t),
            }
        results.append(row)
        dice_val = row[f'{muscle_name}_dice']; hd_val = row[f'{muscle_name}_hausdorff']
        print(f'  Aug_{idx:>3s}  dice={dice_val:.4f}  hd={hd_val:.2f}mm')
    df = pd.DataFrame(results)
    csv_path = os.path.join(RESULT_DIR, f'df_{muscle_name}_{ALGO_TAG}_sheffield.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows -> {csv_path}')
    return df


dfs = {}
for muscle_name, sheffield_label, npz_keys in MUSCLES:
    print(f'\n-- {muscle_name}  (GT={sheffield_label}) --')
    dfs[muscle_name] = evaluate_muscle(muscle_name, sheffield_label, npz_keys)
print('\nDone.')

In [ ]:
summary_rows = []
for muscle_name, df in dfs.items():
    if df.empty: continue
    summary_rows.append({
        'muscle': muscle_name, 'n': len(df),
        'dice_mean':      df[f'{muscle_name}_dice'].mean(),
        'dice_std':       df[f'{muscle_name}_dice'].std(),
        'hausdorff_mean': df[f'{muscle_name}_hausdorff'].mean(),
        'hausdorff_std':  df[f'{muscle_name}_hausdorff'].std(),
    })
summary      = pd.DataFrame(summary_rows).set_index('muscle')
summary_path = os.path.join(RESULT_DIR, f'summary_{ALGO_TAG}_sheffield.csv')
summary.to_csv(summary_path)
print(f'Summary saved -> {summary_path}\n')
print(summary.round(4).to_string())